# Bivariate Analysis

**Topic:** Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, HBox, VBox, HTML

from IPython.display import display, clear_output
from scipy import stats

titanic = sns.load_dataset("titanic")
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this session you will be able to:

- **Select** the correct chart type when comparing two variables based on their types
- **Interpret** a scatter plot, box plot, and grouped bar chart in the context of the Titanic data
- **Identify** which variables show the strongest relationship with the survival outcome

> **Tip:** Start with `pclass` as X and `survived` as Y. That gives you a grouped bar chart. Then try `age` as X and `survived` as Y for a box plot. Then try `age` and `fare` together for a scatter plot with a correlation coefficient.

---
## How we got here

In `08_univariate_analysis.ipynb` we looked at each variable individually. Now we compare pairs. Bivariate analysis is where we start finding answers to our central question: what predicts survival? This builds directly on the correlation and covariance concepts from `statistics/07_correlation_covariance.ipynb`.

---
## Why this matters for data science

Bivariate analysis reveals which features are likely to be useful predictors before you train a single model. A feature that shows a clear, consistent difference in outcome between groups is a candidate for your model. A feature that shows no relationship with the outcome is probably not worth including. EDA-driven feature selection saves computation time and often produces better-performing models than throwing everything in.

---
## Try it yourself

In [ ]:
out = Output()

usable_cols = [c for c in titanic.columns if c not in ["name", "ticket", "cabin", "boat", "body", "home.dest"]]
_all_numeric = titanic.select_dtypes(include="number").columns.tolist()
numeric_cols = [c for c in _all_numeric if titanic[c].nunique() > 10]
categorical_cols = [c for c in usable_cols if c not in numeric_cols]

x_dropdown = Dropdown(
    options=usable_cols,
    value="pclass",
    description="X / Group:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="300px"),
)

y_dropdown = Dropdown(
    options=usable_cols,
    value="survived",
    description="Y / Value:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="300px"),
)

def render(x_col, y_col):
    if x_col == y_col:
        with out:
            clear_output(wait=True)
            print("Select two different columns to see a comparison.")
        return

    x_is_num = x_col in numeric_cols
    y_is_num = y_col in numeric_cols

    df = titanic[[x_col, y_col]].dropna()

    if x_is_num and y_is_num:
        r, p = stats.pearsonr(df[x_col], df[y_col])
        title = f"{x_col} vs {y_col} — Scatter  (r = {r:.3f}, p = {p:.3f})"
        traces = [go.Scatter(
            x=df[x_col], y=df[y_col],
            mode="markers",
            marker=dict(color=PALETTE["primary"], opacity=0.4, size=6),
            name="Passengers",
        )]
        layout = base_layout(title=title, xaxis_title=x_col, yaxis_title=y_col)

    elif x_is_num and not y_is_num:
        groups = df[y_col].unique()
        colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"], PALETTE["muted"]]
        traces = [
            go.Box(
                x=df.loc[df[y_col] == g, x_col],
                name=str(g),
                marker_color=colors[i % len(colors)],
                boxpoints="outliers",
            )
            for i, g in enumerate(groups)
        ]
        title = f"{x_col} by {y_col} — Box Plot"
        layout = base_layout(title=title, xaxis_title=x_col, yaxis_title="")

    elif not x_is_num and y_is_num:
        groups = df[x_col].unique()
        colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"], PALETTE["muted"]]
        traces = [
            go.Box(
                x=df.loc[df[x_col] == g, y_col],
                name=str(g),
                marker_color=colors[i % len(colors)],
                boxpoints="outliers",
            )
            for i, g in enumerate(groups)
        ]
        title = f"{y_col} by {x_col} — Box Plot"
        layout = base_layout(title=title, xaxis_title=y_col, yaxis_title="")

    else:
        pivot = df.groupby([x_col, y_col]).size().unstack(fill_value=0)
        colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"], PALETTE["muted"]]
        traces = [
            go.Bar(
                name=str(col),
                x=[str(v) for v in pivot.index],
                y=pivot[col].values,
                marker_color=colors[i % len(colors)],
            )
            for i, col in enumerate(pivot.columns)
        ]
        title = f"{x_col} vs {y_col} — Grouped Bar"
        layout = base_layout(title=title, xaxis_title=x_col, yaxis_title="Count")
        layout.update(barmode="group")

    layout.update(showlegend=True, height=420)
    fig = go.Figure(data=traces, layout=layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

def on_change(change):
    render(x_dropdown.value, y_dropdown.value)

x_dropdown.observe(on_change, names="value")
y_dropdown.observe(on_change, names="value")

display(VBox([HBox([x_dropdown, y_dropdown]), out]))
render(x_dropdown.value, y_dropdown.value)

---
## What's happening?

The right chart depends on the types of both columns:

| X column | Y column | Chart | What it shows |
|----------|----------|-------|--------------|
| Numeric | Numeric | Scatter plot | Direction and strength of linear relationship |
| Categorical | Numeric | Box plot | Distribution of Y within each group of X |
| Numeric | Categorical | Box plot | Distribution of X within each group of Y |
| Categorical | Categorical | Grouped bar | Count of combinations |

Correlation (from `statistics/07_correlation_covariance.ipynb`) quantifies the linear relationship between two numeric variables. The scatter plot displays the raw data; the r value in the title gives you the number. When comparing a numeric column to a categorical one, the box plot shows whether the distributions in different groups overlap or are clearly separated.

---
## Real-world example: Titanic Dataset

The chart below shows survival rate broken down by passenger class. This is a categorical vs. numeric bivariate comparison.

Notice:
- **1st class passengers** had a 63% survival rate
- **2nd class passengers** had a 47% survival rate
- **3rd class passengers** had only a 24% survival rate

The survival gap between 1st and 3rd class is enormous. This is one of the most important bivariate relationships in the entire dataset and a likely candidate for any predictive model.

> **Discussion question:** Which single variable is the strongest predictor of survival: passenger class, sex, or age? How can you tell from the charts? Try all three in the widget above before answering.

In [ ]:
survival_by_class = titanic.groupby("pclass")["survived"].mean().reset_index()
survival_by_class.columns = ["pclass", "survival_rate"]
survival_by_class["pct_label"] = (survival_by_class["survival_rate"] * 100).round(1).astype(str) + "%"
survival_by_class["class_label"] = survival_by_class["pclass"].map({1: "1st Class", 2: "2nd Class", 3: "3rd Class"})

colors = [PALETTE["accent"], PALETTE["primary"], PALETTE["secondary"]]

fig = go.Figure(
    data=[go.Bar(
        x=survival_by_class["class_label"],
        y=survival_by_class["survival_rate"] * 100,
        marker_color=colors,
        text=survival_by_class["pct_label"],
        textposition="outside",
        width=0.5,
    )],
    layout=base_layout(
        title="Survival Rate by Passenger Class (Titanic)",
        xaxis_title="Passenger Class",
        yaxis_title="Survival Rate (%)",
    ),
)
fig.update_layout(showlegend=False, yaxis=dict(range=[0, 80]))
fig.show()

### Chart selection guide for bivariate analysis

| Variable pair | Best chart | Watch out for |
|---------------|-----------|---------------|
| Age vs fare (numeric/numeric) | Scatter | Overplotting with 800+ points — use opacity |
| Class vs age (cat/num) | Box plot | Few categories make box plot work well |
| Sex vs survival (cat/cat) | Grouped or stacked bar | Normalize to percentages for fair comparison |
| Fare vs survival (num/cat) | Box plot or violin | Outlier fares can compress the main chart area |
| Embarked vs pclass (cat/cat) | Heatmap or grouped bar | Many combinations — heatmap scales better |

---
## Key takeaway

> **The chart type is not a preference. It is determined by the variable types. Choosing the wrong chart for two variables is like reporting a median for a categorical column — the output has no meaning.**

---
*Next up: 10_multivariate_analysis.ipynb — looking at multiple variables at once using correlation heatmaps and grouped visualizations*